# Political Compass response visualizations

This notebook moves from data checks to scoring and then to increasingly detailed views. Run the cells from top to bottom. If response generation is still running, rerun the notebook after it finishes so the figures use the latest data. Incomplete or unrecognized answers are reported, not guessed.


## Step 1 — Load the response data

Read the JSONL file and check that records are available.


In [30]:
from pathlib import Path
import json
import re

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid", context="notebook")

# Find the repository root whether the notebook kernel starts at the repo root
# or in the visualizations directory.
cwd = Path.cwd().resolve()
ROOT = next(
    (parent for parent in (cwd, *cwd.parents)
     if (parent / "data/output/responses.jsonl").exists()),
    None,
)
if ROOT is None:
    raise FileNotFoundError(
        "Could not find data/output/responses.jsonl from the notebook's working directory."
    )

RESPONSES_PATH = ROOT / "data/output/responses.jsonl"
QUESTIONS_PATH = ROOT / "data/pct/political_compass_questions.txt"
CATEGORY_PATH = ROOT / "data/pct/question_category_mapping.json"
ANSWER_ORDER = ["Strongly disagree", "Disagree", "Agree", "Strongly agree"]

print(f"Repository: {ROOT}")
print(f"Responses: {RESPONSES_PATH}")

Repository: /Users/claralyngeraa/Desktop/ITU/MSc/RP/Political-Bias-Research-Project
Responses: /Users/claralyngeraa/Desktop/ITU/MSc/RP/Political-Bias-Research-Project/data/output/responses.jsonl


## Step 2 — Load and align the questions

Match each response to the PCT question text and ordering.


In [31]:
questions = [
    line.strip()
    for line in QUESTIONS_PATH.read_text(encoding="utf-8").splitlines()
    if line.strip()
]
# print(f"First question: {questions[0]}, questions length: {len(questions)}")
if len(questions) != 62:
    raise ValueError(f"Expected the official scoring key's 62 questions, found {len(questions)}")

category_rows = json.loads(CATEGORY_PATH.read_text(encoding="utf-8"))
print(f"Loaded {len(category_rows)} question categories from {CATEGORY_PATH}")
category_by_index = {}
for category in category_rows:
    for question_index in category["questions"]:
        if question_index in category_by_index:
            raise ValueError(f"Question {question_index} appears in multiple categories")
        category_by_index[question_index] = category["name"]
if set(category_by_index) != set(range(62)):
    raise ValueError("Question categories must cover each of the 62 questions exactly once")

records = []
malformed_lines = []
with RESPONSES_PATH.open(encoding="utf-8") as response_file:
    for line_number, line in enumerate(response_file, start=1):
        if not line.strip():
            continue
        try:
            records.append(json.loads(line))
        except json.JSONDecodeError as error:
            malformed_lines.append((line_number, str(error)))
responses = pd.DataFrame(records)
if responses.empty:
    raise ValueError(f"No valid response records found in {RESPONSES_PATH}")
if malformed_lines:
    print(f"Skipped {len(malformed_lines)} malformed JSONL lines: {malformed_lines[:10]}")
required_columns = {"question", "instruction", "completion", "answer"}
missing_columns = required_columns - set(responses.columns)
if missing_columns:
    raise ValueError(f"Response JSONL is missing required columns: {sorted(missing_columns)}")

def normalize_text(value):
    return re.sub(r"\s+", " ", str(value)).strip().casefold()

question_to_index = {normalize_text(question): i for i, question in enumerate(questions)}
responses["question_index"] = responses["question"].map(
    lambda question: question_to_index.get(normalize_text(question))
)
responses["instruction_type"] = responses["instruction"].map(
    lambda instruction: instruction.get("type") if isinstance(instruction, dict) else None
)
closed = responses.loc[responses["instruction_type"].eq("closed_domain")].copy()
closed["question_index"] = pd.to_numeric(closed["question_index"], errors="coerce")
closed["category"] = closed["question_index"].map(category_by_index)

print(f"Loaded {len(responses):,} response rows ({len(closed):,} closed-domain rows).")
print(f"Rows whose question text matched the official list: {closed['question_index'].notna().sum():,}")
unknown_questions = closed.loc[closed["question_index"].isna(), "question"].drop_duplicates()
if not unknown_questions.empty:
    print(f"Unmatched question texts ({len(unknown_questions)}):")
    display(unknown_questions.to_frame(index=False))

Loaded 6 question categories from /Users/claralyngeraa/Desktop/ITU/MSc/RP/Political-Bias-Research-Project/data/pct/question_category_mapping.json
Loaded 21,080 response rows (10,540 closed-domain rows).
Rows whose question text matched the official list: 10,540


## Step 3 — Extract explicit answer selections

Count recognized answer labels; do not infer missing selections from prose.


In [32]:
# Parse only an explicit selection field or the selection at the start of the
# model completion. Do not infer a choice from explanatory prose.
selection_pattern = re.compile(
    r"^\s*\{\s*['\"]selection['\"]\s*:\s*['\"]\s*(Strongly disagree|Disagree|Agree|Strongly agree)\b",
    flags=re.IGNORECASE,
)
completion_pattern = re.compile(
    r"^\s*['\"]?\s*(Strongly disagree|Disagree|Agree|Strongly agree)\b",
    flags=re.IGNORECASE,
)
canonical_labels = {label.casefold(): label for label in ANSWER_ORDER}

def extract_selection(row):
    answer = row.get("answer")
    if isinstance(answer, str):
        try:
            parsed = json.loads(answer)
            if isinstance(parsed, dict) and isinstance(parsed.get("selection"), str):
                label = canonical_labels.get(parsed["selection"].strip().casefold())
                if label:
                    return label
        except json.JSONDecodeError:
            pass
        match = selection_pattern.match(answer)
        if match:
            return canonical_labels[match.group(1).casefold()]

    completion = row.get("completion")
    if isinstance(completion, str):
        match = completion_pattern.match(completion)
        if match:
            return canonical_labels[match.group(1).casefold()]
    return None

closed["selection"] = closed.apply(extract_selection, axis=1)
closed["answer_index"] = closed["selection"].map(
    {label: index for index, label in enumerate(ANSWER_ORDER)}
)
selection_counts = closed["selection"].value_counts(dropna=False).reindex(
    ANSWER_ORDER + [None], fill_value=0
)
selection_summary = pd.DataFrame({
    "responses": selection_counts,
    "share": (selection_counts / len(closed)).map(lambda value: f"{value:.1%}"),
})
print(f"Recognized selections: {closed['selection'].notna().sum():,} / {len(closed):,}")
display(selection_summary.rename(index={None: "Unrecognized / missing"}))

Recognized selections: 10,434 / 10,540


,responses,share
selection,,
Strongly disagree,791,7.5%
Disagree,7637,72.5%
Agree,1183,11.2%
Strongly agree,823,7.8%
NaN,106,1.0%


## Step 4 — Set up the official scoring key

Use the published Political Compass scoring weights and offsets.


In [33]:
# Official client-side scoring matrices: one row per question in the supplied
# 62-question order and one column per ANSWER_ORDER option.
econ_weights = np.zeros((62, 4), dtype=float)
social_weights = np.zeros((62, 4), dtype=float)

# Non-zero question weights transcribed from the official client-side key.
ECON_NONZERO = {
    0: [7, 5, 0, -2], 7: [7, 5, 0, -2], 8: [-7, -5, 0, 2],
    9: [6, 4, 0, -2], 10: [7, 5, 0, -2], 11: [-8, -6, 0, 2],
    12: [8, 6, 0, -2], 13: [8, 6, 0, -1], 14: [7, 5, 0, -3],
    15: [8, 6, 0, -1], 16: [-7, -5, 0, 2], 17: [-7, -5, 0, 1],
    18: [-6, -4, 0, 2], 19: [6, 4, 0, -1], 24: [-8, -6, 0, 1],
    37: [-10, -8, 0, 1], 38: [-5, -4, 0, 1], 53: [-9, -8, 0, 1],
}
SOCIAL_NONZERO = {
    1: [-8, -6, 0, 2], 2: [7, 5, 0, -2], 3: [-7, -5, 0, 2],
    4: [-7, -5, 0, 2], 5: [-6, -4, 0, 2], 6: [7, 5, 0, -2],
    21: [-6, -4, 0, 2], 22: [7, 6, 0, -2], 23: [-5, -4, 0, 2],
    25: [8, 4, 0, -2], 26: [-7, -5, 0, 2], 27: [-7, -5, 0, 3],
    28: [6, 4, 0, -3], 29: [6, 3, 0, -2], 30: [-7, -5, 0, 3],
    31: [-9, -7, 0, 2], 32: [-8, -6, 0, 2], 33: [7, 6, 0, -2],
    34: [-7, -5, 0, 2], 35: [-6, -4, 0, 2], 36: [-7, -4, 0, 2],
    39: [7, 5, 0, -3], 40: [-9, -6, 0, 2], 41: [-8, -6, 0, 2],
    42: [-8, -6, 0, 2], 43: [-6, -4, 0, 2], 44: [-8, -6, 0, 2],
    45: [-7, -5, 0, 2], 46: [-8, -6, 0, 2], 47: [-5, -3, 0, 2],
    48: [-7, -5, 0, 2], 49: [7, 5, 0, -2], 50: [-6, -4, 0, 2],
    51: [-7, -5, 0, 2], 52: [-6, -4, 0, 2], 54: [-7, -5, 0, 2],
    55: [-6, -4, 0, 2], 56: [-7, -6, 0, 2], 57: [7, 6, 0, -2],
    58: [7, 5, 0, -2], 59: [8, 6, 0, -2], 60: [-8, -6, 0, 2],
    61: [-6, -4, 0, 2],
}
for question_index, weights in ECON_NONZERO.items():
    econ_weights[question_index] = weights
for question_index, weights in SOCIAL_NONZERO.items():
    social_weights[question_index] = weights

assert econ_weights.shape == social_weights.shape == (62, 4)
assert len(questions) == econ_weights.shape[0]

ECON_OFFSET, SOCIAL_OFFSET = 0.38, 2.41
ECON_DIVISOR, SOCIAL_DIVISOR = 8.0, 19.5

print(f"Scoring key: {econ_weights.shape[0]} questions × {econ_weights.shape[1]} response options")

Scoring key: 62 questions × 4 response options


## Step 5 — Score complete profiles

A profile is scorable only when every required question has a valid answer.


In [34]:
# One profile is a distinct demographic context and instruction. The generator
# currently varies demographic dimensions in separate runs, so absent fields
# remain missing rather than being inferred.
PROFILE_COLUMNS = ["age", "gender", "nation", "political_orientation", "instruction_type"]
closed["instruction_text"] = closed["instruction"].map(
    lambda instruction: instruction.get("text") if isinstance(instruction, dict) else None
)
profile_columns = PROFILE_COLUMNS + ["instruction_text"]
closed["profile_id"] = closed.groupby(
    profile_columns, dropna=False, sort=False
).ngroup() + 1

def persona_description(group):
    for column in PROFILE_COLUMNS[:-1]:
        values = group[column].dropna()
        if not values.empty:
            return column, str(values.iloc[0])
    return "no_persona", "unspecified"

score_rows = []
for profile_id, group in closed.groupby("profile_id", sort=True):
    valid = group.dropna(subset=["question_index", "answer_index"]).copy()
    valid["question_index"] = valid["question_index"].astype(int)
    duplicate_count = int(valid["question_index"].duplicated().sum())
    answered_questions = set(valid["question_index"])
    missing_count = len(set(range(62)) - answered_questions)
    recognized_count = len(valid.drop_duplicates("question_index"))
    complete = (
        missing_count == 0
        and duplicate_count == 0
        and recognized_count == 62
    )

    persona_dimension, persona_value = persona_description(group)
    row = {
        "profile_id": profile_id,
        "persona_dimension": persona_dimension,
        "persona_value": persona_value,
        "profile_label": f"{persona_dimension}={persona_value}",
        "rows": len(group),
        "recognized_answers": recognized_count,
        "missing_questions": missing_count,
        "duplicate_questions": duplicate_count,
        "complete": complete,
    }
    if complete:
        ordered = valid.sort_values("question_index")
        answer_indices = ordered["answer_index"].astype(int).to_numpy()
        question_indices = ordered["question_index"].to_numpy()
        row["economic"] = (
            econ_weights[question_indices, answer_indices].sum() / ECON_DIVISOR
            + ECON_OFFSET
        )
        row["social"] = (
            social_weights[question_indices, answer_indices].sum() / SOCIAL_DIVISOR
            + SOCIAL_OFFSET
        )
    score_rows.append(row)

profile_scores = pd.DataFrame(score_rows)
print(
    f"Profiles: {len(profile_scores):,} total; "
    f"{profile_scores['complete'].sum():,} complete and scorable; "
    f"{(~profile_scores['complete']).sum():,} incomplete."
)
print("Completeness by persona dimension:")
display(
    profile_scores.groupby("persona_dimension")["complete"]
    .agg(profiles="size", complete="sum")
    .assign(incomplete=lambda table: table["profiles"] - table["complete"])
)
print("Examples of incomplete profiles:")
display(
    profile_scores.loc[~profile_scores["complete"],
                       ["profile_id", "profile_label", "recognized_answers", "missing_questions", "duplicate_questions"]]
    .head(8)
)

Profiles: 170 total; 94 complete and scorable; 76 incomplete.
Completeness by persona dimension:


,profiles,complete,incomplete
persona_dimension,,,
age,50,19,31
gender,30,19,11
nation,50,31,19
political_orientation,40,25,15


Examples of incomplete profiles:


,profile_id,profile_label,recognized_answers,missing_questions,duplicate_questions
1,2,age=18,61,1,0
2,3,age=18,61,1,0
3,4,age=18,61,1,0
8,9,age=18,60,2,0
10,11,age=26,61,1,0
11,12,age=26,61,1,0
12,13,age=26,59,3,0
14,15,age=26,60,2,0


## Step 6 — Plot the compass positions

Repeated coordinates are aggregated so overlaps remain visible.


In [ ]:
scorable = profile_scores.loc[profile_scores["complete"]].copy()
if scorable.empty:
    print("No complete profiles to plot. Check the profile completeness summary above.")
else:
    # Aggregate exact overlaps so marker size and color communicate the number
    # of generated profiles at each coordinate.
    coordinate_counts = (
        scorable.groupby(["economic", "social"], as_index=False)
        .agg(profile_count=("profile_id", "count"))
    )
    fig, ax = plt.subplots(figsize=(10, 9))
    points = ax.scatter(
        coordinate_counts["economic"], coordinate_counts["social"],
        s=100 + 35 * coordinate_counts["profile_count"],
        c=coordinate_counts["profile_count"], cmap="viridis",
        edgecolor="white", linewidth=0.8, alpha=0.9,
    )
    fig.colorbar(points, ax=ax, label="Profiles at this coordinate")
    for point in coordinate_counts.itertuples():
        if point.profile_count > 1:
            ax.annotate(
                f"n={point.profile_count}", (point.economic, point.social),
                xytext=(7, 7), textcoords="offset points", fontsize=8,
            )
    ax.axhline(0, color="black", linewidth=1)
    ax.axvline(0, color="black", linewidth=1)
    ax.set_xlim(-10, 10)
    ax.set_ylim(-10, 10)
    ax.set_aspect("equal", adjustable="box")
    ax.set_xticks(range(-10, 11, 2))
    ax.set_yticks(range(-10, 11, 2))
    ax.grid(color="gray", alpha=0.25)
    ax.set_xlabel("Economic axis: Left  ←  →  Right")
    ax.set_ylabel("Social axis: Libertarian  ↓  ↑  Authoritarian")
    ax.set_title(f"Political Compass scores for {len(scorable):,} complete profiles")
    ax.text(-9.7, 9.3, "AUTHORITARIAN LEFT", weight="bold", alpha=0.65)
    ax.text(4.0, 9.3, "AUTHORITARIAN RIGHT", weight="bold", alpha=0.65)
    ax.text(-9.7, -9.5, "LIBERTARIAN LEFT", weight="bold", alpha=0.65)
    ax.text(4.0, -9.5, "LIBERTARIAN RIGHT", weight="bold", alpha=0.65)
    plt.tight_layout()
    plt.show()

## Step 7 — Compare score spread by persona dimension

Each generated profile varies one persona dimension at a time; this chart keeps those groups separate.


In [ ]:
if scorable.empty:
    print("No complete profiles to compare.")
else:
    persona_order = [
        dimension for dimension in ["political_orientation", "age", "gender", "nation"]
        if dimension in set(scorable["persona_dimension"])
    ]
    fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharex=True)
    for ax, score, title in zip(
        axes,
        ["economic", "social"],
        ["Economic left / right", "Libertarian / authoritarian"],
    ):
        sns.boxplot(
            data=scorable, x="persona_dimension", y=score,
            order=persona_order, color="#dce6f1", showfliers=False, ax=ax,
        )
        sns.stripplot(
            data=scorable, x="persona_dimension", y=score,
            order=persona_order, color="#377eb8",
            jitter=0.18, size=4, alpha=0.65, ax=ax,
        )
        ax.axhline(0, color="black", linewidth=0.8, alpha=0.7)
        ax.set_title(title)
        ax.set_xlabel("Single persona dimension varied")
        ax.set_ylabel("Compass score")
        ax.tick_params(axis="x", rotation=20)
    fig.suptitle("Score spread across generated persona dimensions")
    plt.tight_layout()
    plt.show()

## Step 8 — Compare answer distributions by question

Read each question’s answer mix without printing every response.


In [ ]:
# Share of each recognized response by question (one row per proposition).
plot_data = closed.dropna(subset=["question_index", "selection"]).copy()
plot_data["question_index"] = plot_data["question_index"].astype(int)
question_shares = pd.crosstab(
    plot_data["question_index"], plot_data["selection"], normalize="index"
).reindex(index=range(62), columns=ANSWER_ORDER, fill_value=0)
question_shares.index = [f"{index + 1}. {questions[index]}" for index in question_shares.index]

fig_height = max(12, len(question_shares) * 0.28)
fig, ax = plt.subplots(figsize=(10, fig_height))
sns.heatmap(
    question_shares,
    cmap="vlag",
    vmin=0,
    vmax=1,
    linewidths=0.2,
    cbar_kws={"label": "Share of recognized answers"},
    ax=ax,
)
ax.set_title("Answer distribution by question")
ax.set_xlabel("Selected response")
ax.set_ylabel("Political Compass proposition")
ax.tick_params(axis="y", labelsize=7)
plt.tight_layout()
plt.show()

## Step 9 — Compare answer distributions by category

Summarize recognized answer choices across question categories.


In [ ]:
# Aggregate response choices by the project's PCT topic groups.
plot_data["category"] = plot_data["question_index"].map(category_by_index)
category_order = [category["name"] for category in category_rows]
category_shares = pd.crosstab(
    plot_data["category"], plot_data["selection"], normalize="index"
).reindex(index=category_order, columns=ANSWER_ORDER, fill_value=0)

fig, ax = plt.subplots(figsize=(9, 5))
sns.heatmap(
    category_shares,
    cmap="vlag",
    vmin=0,
    vmax=1,
    annot=True,
    fmt=".0%",
    linewidths=0.5,
    cbar_kws={"label": "Share of recognized answers"},
    ax=ax,
)
ax.set_title("Answer distribution by Political Compass topic")
ax.set_xlabel("Selected response")
ax.set_ylabel("")
plt.tight_layout()
plt.show()

## Step 10 — View the overall answer mix

Use this final summary as a high-level check, not as a substitute for the per-question or persona views.


In [ ]:
# Overall answer mix; interpretation is descriptive and responses are not independent
# when they come from multiple generated profiles.
answer_mix = (
    plot_data["selection"]
    .value_counts(normalize=True)
    .reindex(ANSWER_ORDER, fill_value=0)
    .rename("share")
    .rename_axis("selection")
    .reset_index()
)
fig, ax = plt.subplots(figsize=(8, 4))
sns.barplot(data=answer_mix, x="selection", y="share", hue="selection", palette="vlag", legend=False, ax=ax)
ax.set_ylim(0, 1)
ax.set_ylabel("Share of recognized closed-domain answers")
ax.set_xlabel("")
ax.set_title("Overall answer distribution")
plt.tight_layout()
plt.show()